In [2]:
# ===== Task 4 → AGE ONLY: ARIMA → CSVs + Plots (to 2043) =====
# If needed once:  pip install pandas numpy matplotlib statsmodels openpyxl
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ---------- PATHS ----------
DATA_DIR    = Path(r"D:\arima project\Task 4\data")
RESULTS_DIR = Path(r"D:\arima project\Task 4\results\age")  # age outputs here
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
END_YEAR    = 2043

# Optionally force the AAMR column if auto-detect fails (else leave None)
MANUAL_AAMR_COL = None  # e.g., "Age Adjusted Rate"

# ---------- PLOTTING STYLE ----------
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# ---------- HELPERS ----------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0; newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns:
        return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group(df):
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    for pat in [r"\bage\s*group(s)?\b|age\s*cat|age\-group", r"\bvariable\b"]:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    return None  # overall-only sheet

def choose_age_sheet(xls):
    # prefer “final” + “age” if present
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "final" in s: sc += 5
        if "age"   in s: sc += 4
        if "data"  in s: sc += 2
        return sc
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def load_observed_excel(path: Path, sheet_name: str|None, overall_label="Overall"):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_age_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol and gcol == tcol:
        gcol = None

    if gcol is None:
        tidy = (df[[ycol, tcol]].dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = overall_label
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (df[[ycol, gcol, tcol]].dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or overall_label)}'")
    print(tidy.head(6))
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0): 
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# ---------- RUN FOR AGE ----------
# Pick the first Excel whose file name contains "AGE"
age_file = None
for p in sorted(DATA_DIR.glob("*.xls*")):
    if "age" in p.name.lower():
        age_file = p; break
if age_file is None:
    raise FileNotFoundError("No age Excel found in data folder.")

print("Using age workbook:", age_file)

# Load observed age series (overall and/or subgroups)
observed = load_observed_excel(age_file, sheet_name=None, overall_label="Overall")
groups   = sorted(observed["Group"].unique())
print("Detected age groups:", groups)

# Fit each group, save CSV + individual plot
all_rows = []
last_obs_year = int(observed["Year"].max())
for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    order, fc = forecast_to(y, end_year=END_YEAR, conf=0.95)

    # Save per-group CSV (2 decimals)
    csv_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # Keep for combined CSV
    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

    # Per-group plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs)")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)     # clip CI at 0 for display only
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)

    ax.set_title(f"Age-group {g} — observed (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = RESULTS_DIR / f"{safe_name(g)}_timeseries_to_{END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

# Combined CSV for all age groups
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv", index=False)
    print("Consolidated CSV:", RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv")

# Combined plot (all age groups together)
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Age-group (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = RESULTS_DIR / f"COMBINED_timeseries_to_{END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", RESULTS_DIR)


Using age workbook: D:\arima project\Task 4\data\Copy of AGE GOUP COMLETE DM & STROKE(1).xlsx

Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'variable'
   Year          Group   AAMR
0  1999  middle adults  41.56
1  2000  middle adults  42.91
2  2001  middle adults  41.30
3  2002  middle adults  39.00
4  2003  middle adults  37.75
5  2004  middle adults  33.13
Detected age groups: ['middle adults', 'old adults', 'young adults']
[middle adults] ARIMA order=(2, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\age\middle_adults_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\age\middle_adults_timeseries_to_2043.png
[old adults] ARIMA order=(0, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\age\old_adults_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\age\old_adults_timeseries_to_2043.png
[young adults] ARIMA order=(1, 0, 3, 'c') → CSV: D:\arima project\Task 4\results\age\young_adults_forecast_to_2043.csv
   Plot: D:\arima pr

In [4]:
# ===== Task 4 → REGION ONLY: ARIMA → CSVs + Plots (to 2043) =====
# One-time (if needed):  pip install pandas numpy matplotlib statsmodels openpyxl
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ---------- PATHS ----------
DATA_DIR    = Path(r"D:\arima project\Task 4\data")
RESULTS_DIR = Path(r"D:\arima project\Task 4\results\region")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
END_YEAR    = 2043

# If auto-detect fails, force the exact AAMR column name here (else leave None)
MANUAL_AAMR_COL = None  # e.g., "Age Adjusted Rate"

# ---------- PLOTTING STYLE ----------
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# ---------- HELPERS ----------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0; newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns:
        return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    # fallback: "age"+"adjust" and "rate" present (but not "group")
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group_region(df):
    """Find Census Region group column."""
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    for pat in [r"\bcensus\s*region\b", r"\bregion\b", r"\bdivision\b"]:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    # sometimes authors store it under a generic 'variable'
    c = _pick_col(df, [r"\bvariable\b"], required=False, exclude_regex=exclude)
    return c

def choose_region_sheet(xls):
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "final"  in s: sc += 5
        if "census" in s or "region" in s: sc += 4
        if "data"   in s: sc += 2
        return sc
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def load_observed_excel(path: Path, sheet_name: str|None, overall_label="Overall"):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_region_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group_region(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol and gcol == tcol:
        gcol = None

    if gcol is None:
        tidy = (df[[ycol, tcol]].dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = overall_label
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (df[[ycol, gcol, tcol]].dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or overall_label)}'")
    print(tidy.head(6))
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0):
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# ---------- RUN FOR REGION ----------
# Pick the first Excel whose file name contains "REGION" or "CENSUS"
region_file = None
for p in sorted(DATA_DIR.glob("*.xls*")):
    nm = p.name.lower()
    if "region" in nm or "census" in nm:
        region_file = p; break
if region_file is None:
    raise FileNotFoundError("No region/census Excel found in the data folder.")

print("Using region workbook:", region_file)

# Load observed (overall/by-region)
observed = load_observed_excel(region_file, sheet_name=None, overall_label="Overall")
groups   = sorted(observed["Group"].unique())
print("Detected regions:", groups)

# Fit each region, save CSV + individual plot
all_rows = []
last_obs_year = int(observed["Year"].max())
for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    order, fc = forecast_to(y, end_year=END_YEAR, conf=0.95)

    # per-region CSV (2 decimals)
    csv_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # stash for consolidated
    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

    # per-region plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs)")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)     # display only
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)

    ax.set_title(f"Region {g} — observed (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = RESULTS_DIR / f"{safe_name(g)}_timeseries_to_{END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

# consolidated CSV
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv", index=False)
    print("Consolidated CSV:", RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv")

# combined plot (all regions)
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Region (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = RESULTS_DIR / f"COMBINED_timeseries_to_{END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", RESULTS_DIR)


Using region workbook: D:\arima project\Task 4\data\Copy of CENSUS REGION COMPLETE DM& STROKE(1).xlsx

Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'Census Region'
   Year                       Group    AAMR
0  1999  Census Region 1: Northeast  104.46
1  2000  Census Region 1: Northeast  101.23
2  2001  Census Region 1: Northeast   98.71
3  2002  Census Region 1: Northeast   94.70
4  2003  Census Region 1: Northeast   87.20
5  2004  Census Region 1: Northeast   81.72
Detected regions: ['Census Region 1: Northeast', 'Census Region 2: Midwest', 'Census Region 3: South', 'Census Region 4: West']
[Census Region 1: Northeast] ARIMA order=(0, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\region\Census_Region_1_Northeast_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\region\Census_Region_1_Northeast_timeseries_to_2043.png
[Census Region 2: Midwest] ARIMA order=(1, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\region\Census_Region_2_Midw

In [4]:
# ===== Task 4 → GENDER ONLY: ARIMA → CSVs + Plots (to 2043) =====
# One-time (if needed):  pip install pandas numpy matplotlib statsmodels openpyxl
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ---------- PATHS ----------
DATA_DIR    = Path(r"D:\arima project\Task 4\data")
RESULTS_DIR = Path(r"D:\arima project\Task 4\results\gender")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
END_YEAR    = 2043

# If auto-detect fails, force the exact AAMR column name here (else leave None)
MANUAL_AAMR_COL = None  # e.g., "Age Adjusted Rate"

# ---------- PLOTTING STYLE ----------
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# ---------- HELPERS ----------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0; newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns:
        return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    # fallback: "age"+"adjust" and "rate" present (but not "group")
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group_gender(df):
    """Find Sex/Gender group column."""
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    for pat in [r"\bsex\b", r"\bgender\b"]:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    # sometimes stored as 'variable'
    return _pick_col(df, [r"\bvariable\b"], required=False, exclude_regex=exclude)

def choose_gender_sheet(xls):
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "final"  in s: sc += 5
        if "gender" in s or "sex" in s: sc += 4
        if "data"   in s: sc += 2
        return sc
    # fall back to any sheet if nothing matches
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def load_observed_excel(path: Path, sheet_name: str|None, overall_label="Overall"):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_gender_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group_gender(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol and gcol == tcol:
        gcol = None

    if gcol is None:
        tidy = (df[[ycol, tcol]].dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = overall_label
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (df[[ycol, gcol, tcol]].dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or overall_label)}'")
    print(tidy.head(6))
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0):
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# ---------- RUN FOR GENDER ----------
# Pick the first Excel whose filename contains "GENDER" or "SEX"
gender_file = None
for p in sorted(DATA_DIR.glob("*.xls*")):
    nm = p.name.lower()
    if "gender" in nm or "sex" in nm:
        gender_file = p; break
if gender_file is None:
    raise FileNotFoundError("No gender/sex Excel found in the data folder.")

print("Using gender workbook:", gender_file)

# Load observed (by Female/Male; “Overall” if no group col)
observed = load_observed_excel(gender_file, sheet_name=None, overall_label="Overall")
groups   = sorted(observed["Group"].unique())
print("Detected gender groups:", groups)

# Fit each group, save CSV + individual plot
all_rows = []
last_obs_year = int(observed["Year"].max())
for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    order, fc = forecast_to(y, end_year=END_YEAR, conf=0.95)

    # per-group CSV (2 decimals)
    csv_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # stash for consolidated
    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

    # per-group plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs)")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)     # display only
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)

    ax.set_title(f"Gender {g} — observed (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = RESULTS_DIR / f"{safe_name(g)}_timeseries_to_{END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

# consolidated CSV
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv", index=False)
    print("Consolidated CSV:", RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv")

# combined plot (Female + Male)
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Gender (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = RESULTS_DIR / f"COMBINED_timeseries_to_{END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", RESULTS_DIR)


Using gender workbook: D:\arima project\Task 4\data\Copy of GENDER COMLETE DM & STROKE(1).xlsx

Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'Variable'
   Year   Group    AAMR
0  1999  Female  124.27
1  2000  Female  121.22
2  2001  Female  118.90
3  2002  Female  116.81
4  2003  Female  108.13
5  2004  Female  100.81
Detected gender groups: ['Female', 'Male']
[Female] ARIMA order=(0, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\gender\Female_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\gender\Female_timeseries_to_2043.png
[Male] ARIMA order=(2, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\gender\Male_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\gender\Male_timeseries_to_2043.png
Consolidated CSV: D:\arima project\Task 4\results\gender\ALL_GROUPS_forecasts_to_2043.csv
Combined plot: D:\arima project\Task 4\results\gender\COMBINED_timeseries_to_2043.png

Done → D:\arima project\Task 4\results\gender


In [6]:
# ===== Task 4 → RACE ONLY: ARIMA → CSVs + Plots (to 2043) =====
# If needed once:  pip install pandas numpy matplotlib statsmodels openpyxl
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# -------- PATHS --------
DATA_DIR    = Path(r"D:\arima project\Task 4\data")
RESULTS_DIR = Path(r"D:\arima project\Task 4\results\race")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
END_YEAR    = 2043

# If auto-detection ever guesses the wrong AAMR column, set this:
MANUAL_AAMR_COL = None   # e.g., "Age Adjusted Rate"

# -------- PLOTTING STYLE --------
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# -------- HELPERS --------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0; newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns:
        return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    # fallback: "age"+"adjust" and "rate" present (but not "group")
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group_race(df):
    """Race / Ethnicity column (Race, Hispanic Origin, Race/Ethnicity, etc.)."""
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    for pat in [r"\brace\b", r"\bhispanic\b", r"\borigin\b", r"race.?/?ethnic", r"\bethnic"]:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    return _pick_col(df, [r"\bvariable\b"], required=False, exclude_regex=exclude)

def choose_race_sheet(xls):
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "final" in s: sc += 5
        if "race"  in s or "hispanic" in s or "origin" in s or "ethnic" in s: sc += 4
        if "data"  in s: sc += 2
        return sc
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def load_observed_excel(path: Path, sheet_name: str|None, overall_label="Overall"):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_race_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group_race(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol and gcol == tcol:
        gcol = None

    if gcol is None:
        tidy = (df[[ycol, tcol]].dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = overall_label
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (df[[ycol, gcol, tcol]].dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or overall_label)}'")
    print(tidy.head(6))
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0):
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# -------- RUN FOR RACE --------
# Pick the first Excel whose filename suggests race/ethnicity
race_file = None
for p in sorted(DATA_DIR.glob("*.xls*")):
    nm = p.name.lower()
    if "race" in nm or "hispanic" in nm or "origin" in nm or "ethnic" in nm:
        race_file = p; break
if race_file is None:
    raise FileNotFoundError("No race/ethnicity Excel found in the data folder.")

print("Using race workbook:", race_file)

# Load observed (by race/ethnicity; “Overall” if no group column)
observed = load_observed_excel(race_file, sheet_name=None, overall_label="Overall")
groups   = sorted(observed["Group"].unique())
print("Detected race groups:", groups)

# Fit each group, save CSV + individual plot
all_rows = []
last_obs_year = int(observed["Year"].max())
for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    order, fc = forecast_to(y, end_year=END_YEAR, conf=0.95)

    # per-group CSV (2 decimals)
    csv_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # stash for consolidated
    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

    # per-group plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs)")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)   # display only
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)

    ax.set_title(f"Race {g} (≤{last_obs_year})ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = RESULTS_DIR / f"{safe_name(g)}_timeseries_to_{END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

# consolidated CSV
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv", index=False)
    print("Consolidated CSV:", RESULTS_DIR / f"ALL_GROUPS_forecasts_to_{END_YEAR}.csv")

# combined plot (all race groups together)
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = RESULTS_DIR / f"{safe_name(g)}_forecast_to_{END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Race (≤{last_obs_year}) ARIMA forecast ({last_obs_year+1}–{END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = RESULTS_DIR / f"COMBINED_timeseries_to_{END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", RESULTS_DIR)


Using race workbook: D:\arima project\Task 4\data\Copy of RACE COMPLETE DM & STROKE(1).xlsx

Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'Race'
   Year                             Group    AAMR
0  1999  American Indian or Alaska Native  194.73
1  2000  American Indian or Alaska Native  154.29
2  2001  American Indian or Alaska Native  168.20
3  2002  American Indian or Alaska Native  152.62
4  2003  American Indian or Alaska Native  179.80
5  2004  American Indian or Alaska Native  141.37
Detected race groups: ['American Indian or Alaska Native', 'Asian or Pacific Islander', 'Black or African American', 'Hispanic or Latino', 'White']
[American Indian or Alaska Native] ARIMA order=(0, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\race\American_Indian_or_Alaska_Native_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\race\American_Indian_or_Alaska_Native_timeseries_to_2043.png
[Asian or Pacific Islander] ARIMA order=(0, 2, 3, 'n') → CSV:

In [4]:
# ===== Task 4 → URBANIZATION ONLY: ARIMA → CSVs + Plots (start 2021, to 2043) =====
# If needed once:  pip install pandas numpy matplotlib statsmodels openpyxl
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# -------- PATHS (edit if needed) --------
DATA_DIR    = Path(r"D:\arima project\Task 4\data")
RESULTS_DIR = Path(r"D:\arima project\Task 4\results\overall")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# hard stop for last observed year in Urbanization:
FORCE_LAST_OBS_YEAR = 2020
END_YEAR            = 2043

# If auto-detection ever guesses the wrong AAMR column, set this:
MANUAL_AAMR_COL = None      # e.g., "Age Adjusted Rate"
# If detection fails for group, you can set manually, e.g.:
MANUAL_GROUP_COL = None     # e.g., "Urbanization" or "Urban-Rural Classification"

# -------- PLOTTING STYLE --------
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# -------- HELPERS --------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0; newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns:
        return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group_urban(df):
    if MANUAL_GROUP_COL and MANUAL_GROUP_COL in df.columns:
        return MANUAL_GROUP_COL
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    pats = [
        r"\burbanization\b|\burbanisation\b|\burban[\s\-_/]*rural\b",
        r"\burban\b|\brural\b",
        r"\bmetro\b|\bmetropolitan\b|\bnon[\s\-]*metropolitan\b",
        r"nchs.*urban.*rural", r"\burban[\s\-]*rural.*class",
        r"\bcounty\s*type\b"
    ]
    for pat in pats:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    return _pick_col(df, [r"\bvariable\b"], required=False, exclude_regex=exclude)

def choose_urban_sheet(xls):
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "final" in s: sc += 4
        # prioritize explicit urban/metro wording
        if any(k in s for k in ["urban","rural","metro","metropolitan","urbanization","urbanisation"]): sc += 6
        if "data"  in s: sc += 1
        return sc
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def load_observed_excel(path: Path, sheet_name: str|None, overall_label="Overall"):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_urban_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group_urban(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol and gcol == tcol:
        gcol = None

    if gcol is None:
        tidy = (df[[ycol, tcol]]
                .dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = overall_label
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (df[[ycol, gcol, tcol]]
                .dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    # *** enforce the 2020 cutoff for Urbanization ***
    tidy = tidy[tidy["Year"] <= FORCE_LAST_OBS_YEAR].copy()
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or overall_label)}'")
    print("First rows:\n", tidy.head(6))
    print(f"Last observed year forced to: {FORCE_LAST_OBS_YEAR}")
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0):
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# -------- PICK THE URBANIZATION WORKBOOK --------
urb_file = None
for p in sorted(DATA_DIR.glob("*.xls*")):
    nm = p.name.lower()
    if any(k in nm for k in ["urban", "urbanization", "urbanisation", "metro", "metropolitan"]):
        urb_file = p; break
if urb_file is None:
    # fall back to any excel (you can set explicitly instead)
    cand = list(sorted(DATA_DIR.glob("*.xls*")))
    if not cand:
        raise FileNotFoundError("No Excel files found in DATA_DIR.")
    urb_file = cand[0]

print("Using urbanization workbook:", urb_file)

# -------- LOAD, FIT, SAVE --------
observed = load_observed_excel(urb_file, sheet_name=None, overall_label="Overall")
groups   = sorted(observed["Group"].unique())
print("Detected urbanization groups:", groups)

all_rows = []
last_obs_year = FORCE_LAST_OBS_YEAR   # enforced cutoff

for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    # ensure last index == FORCE_LAST_OBS_YEAR (in case data are sparse)
    if int(y.index.max()) != FORCE_LAST_OBS_YEAR:
        # reindex up to 2020 and interpolate
        full = pd.Index(range(int(y.index.min()), FORCE_LAST_OBS_YEAR+1))
        y = y.reindex(full).interpolate(limit_direction="both")

    order, fc = forecast_to(y, end_year=END_YEAR, conf=0.95)

    # CSV (2 decimals)
    csv_path = RESULTS_DIR / f"{safe_name(g)}_forecast_from_2021_to_{END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # stash for consolidated
    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

    # per-group plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs ≤{last_obs_year})")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.2)

    ax.set_title(f"Urbanization AAMR: {g} — observed (≤{last_obs_year}) & ARIMA forecast (2021–{END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = RESULTS_DIR / f"{safe_name(g)}_timeseries_2021_to_{END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

# consolidated CSV
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(RESULTS_DIR / f"ALL_GROUPS_forecasts_urbanization_2021_to_{END_YEAR}.csv", index=False)
    print("Consolidated CSV:", RESULTS_DIR / f"ALL_GROUPS_forecasts_urbanization_2021_to_{END_YEAR}.csv")

# combined plot
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = RESULTS_DIR / f"{safe_name(g)}_forecast_from_2021_to_{END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=FORCE_LAST_OBS_YEAR+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Urbanization AAMR: observed (≤{FORCE_LAST_OBS_YEAR}) & ARIMA forecast (2021–{END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = RESULTS_DIR / f"COMBINED_timeseries_urbanization_2021_to_{END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", RESULTS_DIR)


Using urbanization workbook: D:\arima project\Task 4\data\Copy of URBANIZATION COMPLETE DM & STROKE(1).xlsx

Sheet: non metro 1999-2020
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'Overall'
First rows:
    Year    Group    AAMR
0  1999  Overall  151.57
1  2000  Overall  147.08
2  2001  Overall  145.16
3  2002  Overall  144.75
4  2003  Overall  134.04
5  2004  Overall  131.43
Last observed year forced to: 2020
Detected urbanization groups: ['Overall']
[Overall] ARIMA order=(2, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\overall\Overall_forecast_from_2021_to_2043.csv
   Plot: D:\arima project\Task 4\results\overall\Overall_timeseries_2021_to_2043.png
Consolidated CSV: D:\arima project\Task 4\results\overall\ALL_GROUPS_forecasts_urbanization_2021_to_2043.csv
Combined plot: D:\arima project\Task 4\results\overall\COMBINED_timeseries_urbanization_2021_to_2043.png

Done → D:\arima project\Task 4\results\overall


In [3]:
# If needed once:
# pip install pandas numpy matplotlib statsmodels openpyxl

import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ========= CONFIG (EDIT PATHS) =========
FILE_PATH  = Path(r"D:\arima project\Task 4\data\Copy of URBANIZATION COMPLETE DM & STROKE(1).xlsx")
SHEET_NAME = None   # None = auto-pick the best "urban/metro" sheet; set exact name if you prefer
OUTPUT_DIR = Path(r"D:\arima project\Task 4\results\urbanization")
LAST_OBS_YEAR     = 2020     # urbanization observed ends at 2020
FORECAST_END_YEAR = 2043     # forecast horizon (starts 2021)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional overrides (use only if auto-detect fails)
MANUAL_AAMR_COL  = None   # e.g., "Age Adjusted Rate"
MANUAL_GROUP_COL = None   # e.g., "Urbanization"

# ========= PLOTTING STYLE =========
plt.rcParams.update({
    "figure.figsize": (13.5, 8.0),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.7,
    "lines.markersize": 6.7,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=15, width=2.0, length=7)
    for s in ax.spines.values():
        s.set_linewidth(2.2)

# ========= HELPERS =========
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if str(x) and str(x) != "nan"]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, newcols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c]+=1; newcols.append(f"{c}.{seen[c]}")
        else:
            seen[c]=0;  newcols.append(c)
    df.columns = newcols
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    if MANUAL_AAMR_COL and MANUAL_AAMR_COL in df.columns: return MANUAL_AAMR_COL
    for p in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [p])
        except: pass
    low = {c: str(c).strip().lower() for c in df.columns}
    for c, lc in low.items():
        if all(k in lc for k in ["age","adjust"]) and "rate" in lc and "group" not in lc:
            return c
    return _pick_col(df, [r"\baamr\b"])

def find_group(df):
    if MANUAL_GROUP_COL and MANUAL_GROUP_COL in df.columns: return MANUAL_GROUP_COL
    exclude = r"age\s*adjust|aamr|rate|ci|conf|se|stderr|mean|median|total|overall|deaths|population"
    pats = [
        r"\burbanization\b|\burbanisation\b|\burban[\s\-_/]*rural\b",
        r"\burban\b|\brural\b",
        r"\bmetro\b|\bmetropolitan\b|\bnon[\s\-]*metropolitan\b",
        r"nchs.*urban.*rural", r"\burban[\s\-]*rural.*class",
        r"\bcounty\s*type\b", r"\bclassification\b",
        r"\bvariable\b"
    ]
    for pat in pats:
        c = _pick_col(df, [pat], required=False, exclude_regex=exclude)
        if c: return c
    return None

def choose_urban_sheet(xls):
    names = xls.sheet_names
    def score(n):
        s = n.lower().strip()
        sc = 0
        if "urban" in s or "metro" in s or "urbanization" in s or "urbanisation" in s: sc += 6
        if "final" in s: sc += 4
        if "data"  in s: sc += 1
        return sc
    return sorted(names, key=lambda nm: (-score(nm), names.index(nm)))[0]

def standardize_group(val: str) -> str:
    s = str(val).strip().lower()
    if any(k in s for k in ["nonmet", "non-met", "non met", "non metropolitan", "non-metropolitan", "rural", "nonmetro"]):
        return "Non-metro"
    if any(k in s for k in ["metro", "metropolitan", "urban"]):
        return "Metro"
    return str(val).strip()

def load_observed_excel(path: Path, sheet_name=None):
    xls = pd.ExcelFile(path)
    chosen = sheet_name or choose_urban_sheet(xls)
    df = pd.read_excel(xls, sheet_name=chosen)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group(df)

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")
    if gcol:
        df[gcol] = df[gcol].astype(str).map(standardize_group)

    if (not gcol) or gcol == tcol:
        tidy = (df[[ycol, tcol]]
                .dropna(subset=[ycol, tcol])
                .groupby(ycol, as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", tcol:"AAMR"}))
        tidy["Group"] = "Overall"
        tidy = tidy[["Year","Group","AAMR"]]
    else:
        tidy = (df[[ycol, gcol, tcol]]
                .dropna(subset=[ycol, gcol, tcol])
                .groupby([ycol, gcol], as_index=False)[tcol].mean()
                .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"}))

    tidy["Year"] = tidy["Year"].astype(int)
    tidy = tidy[tidy["Year"] <= LAST_OBS_YEAR].copy()
    tidy["Group"] = tidy["Group"].map(standardize_group)
    # Keep only Metro / Non-metro
    tidy = tidy[tidy["Group"].isin(["Metro","Non-metro"])].copy()
    tidy = tidy.sort_values(["Group","Year"]).reset_index(drop=True)

    print(f"\nSheet: {chosen}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group: '{(gcol or 'Overall')}'")
    print("Groups found:", sorted(tidy["Group"].unique()))
    print("First rows:\n", tidy.head(6))
    print(f"Last observed year forced to: {LAST_OBS_YEAR}")
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    full = pd.Index(range(int(y.index.min()), LAST_OBS_YEAR+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(4):
            for q in range(4):
                if (p,d,q) == (0,0,0): 
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    aic = res.aic
                    if (best is None) or (aic < best[0]):
                        best = (aic, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps <= 0:
        raise ValueError("Observed already reaches END_YEAR.")
    order, res = select_arima_order(y)
    fc  = res.get_forecast(steps=steps)
    ci  = fc.conf_int(alpha=1-conf)
    yrs = list(range(last_year+1, end_year+1))
    return order, pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# ========= RUN =========
observed = load_observed_excel(FILE_PATH, sheet_name=SHEET_NAME)
groups   = sorted(observed["Group"].unique())
print("Using groups:", groups)   # expect ['Metro', 'Non-metro']

all_rows = []
for g in groups:
    sub = observed[observed["Group"]==g].copy().dropna(subset=["Year","AAMR"]).sort_values("Year")
    y   = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name="AAMR")
    y   = _sanitize_series_for_arima(y)

    order, fc = forecast_to(y, end_year=FORECAST_END_YEAR, conf=0.95)

    # Save CSV (2 decimals)
    csv_path = OUTPUT_DIR / f"{safe_name(g)}_forecast_2021_to_{FORECAST_END_YEAR}.csv"
    out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        out[c] = out[c].round(2)
    out.insert(0, "Series", g)
    out.to_csv(csv_path, index=False)
    print(f"[{g}] ARIMA order={order} → CSV:", csv_path)

    # Save plot (bold, clean, CI shaded)
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label=f"{g} (obs ≤{LAST_OBS_YEAR})")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
    lo = np.maximum(fc["Lo.95"].values, 0.0)  # clip CI at 0
    ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.15, color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=LAST_OBS_YEAR+0.5, linestyle=":", linewidth=2.2)
    ax.set_title(f"Urbanization AAMR — {g}: observed (≤{LAST_OBS_YEAR}) & ARIMA (2021–{FORECAST_END_YEAR})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = OUTPUT_DIR / f"{safe_name(g)}_timeseries_2021_to_{FORECAST_END_YEAR}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

    tmp = out.copy(); tmp["Group"] = g
    all_rows.append(tmp)

# Consolidated CSV
if all_rows:
    all_df = pd.concat(all_rows, ignore_index=True)
    all_df.to_csv(OUTPUT_DIR / f"ALL_urbanization_2021_to_{FORECAST_END_YEAR}.csv", index=False)
    print("Consolidated CSV:", OUTPUT_DIR / f"ALL_urbanization_2021_to_{FORECAST_END_YEAR}.csv")

# Combined plot
fig, ax = plt.subplots()
added_ci = False
for g in groups:
    sub = observed[observed["Group"]==g].copy().sort_values("Year")
    ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
    fc_path = OUTPUT_DIR / f"{safe_name(g)}_forecast_2021_to_{FORECAST_END_YEAR}.csv"
    if fc_path.exists():
        fc = pd.read_csv(fc_path)
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        lo = np.maximum(fc["Lo.95"].values, 0.0)
        if not added_ci:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color(), label="95% CI (fc)")
            added_ci = True
        else:
            ax.fill_between(fc["Year"], lo, fc["Hi.95"].values, alpha=0.12, color=ln.get_color())

ax.axvline(x=LAST_OBS_YEAR+0.5, linestyle=":", linewidth=2.2)
ax.set_title(f"Urbanization AAMR — observed (≤{LAST_OBS_YEAR}) & ARIMA forecast (2021–{FORECAST_END_YEAR})")
ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax)
ax.legend(ncols=2)
ax.margins(x=0.02)
plt.tight_layout()
combined_png = OUTPUT_DIR / f"COMBINED_urbanization_2021_to_{FORECAST_END_YEAR}.png"
plt.savefig(combined_png, dpi=300, bbox_inches="tight"); plt.close()
print("Combined plot:", combined_png)

print("\nDone →", OUTPUT_DIR)


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.



Sheet: non metro 1999-2020
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group: 'Overall'
Groups found: []
First rows:
 Empty DataFrame
Columns: [Year, Group, AAMR]
Index: []
Last observed year forced to: 2020
Using groups: []
Combined plot: D:\arima project\Task 4\results\urbanization\COMBINED_urbanization_2021_to_2043.png

Done → D:\arima project\Task 4\results\urbanization



Sheet: non metro 1999-2020
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Label forced: 'Non-metro'
   Year      Group    AAMR
0  1999  Non-metro  151.57
1  2000  Non-metro  147.08
2  2001  Non-metro  145.16
3  2002  Non-metro  144.75
4  2003  Non-metro  134.04
5  2004  Non-metro  131.43
Last observed year forced to: 2020

Groups detected for modeling: ['Non-metro']
[Non-metro] ARIMA order=(2, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\urbanization\Non-metro_forecast_2021_to_2043.csv
   Plot: D:\arima project\Task 4\results\urbanization\Non-metro_timeseries_2021_to_2043.png
Consolidated CSV: D:\arima project\Task 4\results\urbanization\ALL_urbanization_2021_to_2043.csv
Combined plot: D:\arima project\Task 4\results\urbanization\COMBINED_urbanization_2021_to_2043.png

Done → D:\arima project\Task 4\results\urbanization


In [7]:
# -*- coding: utf-8 -*-
# Urbanization ARIMA (observed ≤2020 → forecast 2021–2043) with proper Metro / Non-metro handling

import re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ========= USER CONFIG =========
FILE_PATH   = Path(r"D:\arima project\Task 4\data\URBANIZATION COMPLETE DM & STROKE(1).xlsx")
OUT_DIR     = Path(r"D:\arima project\Task 4\results\urbanization")

LAST_OBS_YEAR   = 2020         # observed data end
FORECAST_START  = 2021         # first forecast year
FORECAST_END    = 2043         # last forecast year
TITLE_PREFIX    = "Urbanization"
# =================================

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Plot style ----------
plt.rcParams.update({
    "figure.figsize": (13, 7.6),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.6,
    "lines.markersize": 6.5,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=14, width=1.9, length=6)
    for s in ax.spines.values():
        s.set_linewidth(2.0)

# ---------- helpers ----------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if (str(x) != "nan" and str(x).strip() != "")]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, out = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1; out.append(f"{c}.{seen[c]}")
        else:
            seen[c] = 0;  out.append(c)
    df.columns = out
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):  # skip bad matches
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    for pat in [r"\bage[-\s]*adjust(ed)?\s*rate\b", r"\baamr\b", r"\bage[-\s]*standard(ized)?\s*rate\b"]:
        try: return _pick_col(df, [pat])
        except KeyError: pass
    for c in df.columns:
        lc = str(c).lower()
        if ("age" in lc and "adjust" in lc) and ("rate" in lc):
            return c
    return _pick_col(df, [r"\brate\b"])

def find_group_col(df):
    return _pick_col(
        df,
        [r"\burbanization\b", r"\bvariable\b", r"\bgroup\b|category"],
        required=False,
        exclude_regex=r"age\s*adjust|rate|aamr|ci|se|total|overall"
    )

def detect_forced_label_from_sheetname(sheet_name: str) -> str:
    nm = sheet_name.lower()
    if "non" in nm:  # e.g., "non metro 1999-2020"
        return "Non-metro"
    if "met" in nm:  # e.g., "meto 1999-2020", "metro"
        return "Metro"
    return "Urbanization"

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s)).strip("_")

# ---------- loading ----------
def load_sheet_allow_groups(path: Path, sheet_name: str):
    xls = pd.ExcelFile(path)
    df = pd.read_excel(xls, sheet_name=sheet_name)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)
    gcol = find_group_col(df)  # may be None

    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    if gcol:
        df[gcol] = df[gcol].astype(str).str.strip()
        tidy = (
            df[[ycol, gcol, tcol]]
            .dropna(subset=[ycol, gcol, tcol])
            .query(f"{ycol} <= @LAST_OBS_YEAR")
            .groupby([ycol, gcol], as_index=False)[tcol].mean()
            .rename(columns={ycol:"Year", gcol:"Group", tcol:"AAMR"})
        )
        # normalize values to exactly "Metro" / "Non-metro"
        tidy["Group"] = (tidy["Group"]
                         .str.replace(r"(?i)non\s*metro.*", "Non-metro", regex=True)
                         .str.replace(r"(?i)^metro.*", "Metro", regex=True)
                         .str.strip())
    else:
        forced = detect_forced_label_from_sheetname(sheet_name)
        tidy = (
            df[[ycol, tcol]]
            .dropna(subset=[ycol, tcol])
            .query(f"{ycol} <= @LAST_OBS_YEAR")
            .groupby(ycol, as_index=False)[tcol].mean()
            .rename(columns={ycol:"Year", tcol:"AAMR"})
        )
        tidy["Group"] = forced

    # clean and aggregate in case of duplicates
    tidy["Year"] = tidy["Year"].astype(int)
    tidy = (tidy.groupby(["Group","Year"], as_index=False)["AAMR"]
                 .mean()
                 .sort_values(["Group","Year"])
                 .reset_index(drop=True))

    print(f"\nSheet: {sheet_name}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}' | Group column: '{gcol or '(forced from sheet name)'}'")
    print(tidy.head(6))
    print(f"Last observed year forced to: {LAST_OBS_YEAR}")
    return tidy

# ---------- ARIMA ----------
def _sanitize_series_for_arima(y: pd.Series):
    y = y.sort_index()
    # consolidate duplicates defensively
    y = y.groupby(level=0).mean()
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(0,4):
            for q in range(0,4):
                if (p,d,q) == (0,0,0):
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    score = res.aic
                    if (best is None) or (score < best[0]):
                        best = (score, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed series already extends to END_YEAR.")
    order, res = select_arima_order(y)
    fc = res.get_forecast(steps=steps)
    ci = fc.conf_int(alpha=1-conf)
    yrs = np.arange(last_year+1, end_year+1, dtype=int)
    out = pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })
    return order, out

# ---------- Fit / save / plot ----------
def fit_all_groups_and_save(observed_df, out_dir: Path, title_prefix: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    # keep only intended groups
    observed_df["Group"] = observed_df["Group"].replace(
        {r"(?i)non.*metro.*":"Non-metro", r"(?i)^metro$":"Metro"}, regex=True
    )
    observed_df = observed_df[observed_df["Group"].isin(["Metro","Non-metro"])]

    groups = sorted(observed_df["Group"].unique())
    if not groups:
        print("No Metro/Non-metro rows found after normalization.")
        return

    all_rows = []

    for g in groups:
        sub = (observed_df[observed_df["Group"]==g]
               .dropna(subset=["Year","AAMR"])
               .sort_values("Year"))
        # aggregate dup years just in case
        sub = sub.groupby("Year", as_index=False)["AAMR"].mean()

        y = pd.Series(sub["AAMR"].values, index=sub["Year"].astype(int), name=g)
        y = y[y.index <= LAST_OBS_YEAR]
        y = _sanitize_series_for_arima(y)

        order, fc = forecast_to(y, end_year=FORECAST_END, conf=0.95)

        # CSV (2 decimals)
        csv_path = out_dir / f"{safe_name(g)}_forecast_{FORECAST_START}_to_{FORECAST_END}.csv"
        fc_out = fc.copy()
        for c in ["Point.Forecast","Lo.95","Hi.95"]:
            fc_out[c] = fc_out[c].round(2)
        fc_out.insert(0, "Series", g)
        fc_out.to_csv(csv_path, index=False)
        print(f"[{g}] ARIMA order={order} → CSV: {csv_path}")

        # keep for consolidated
        tmp = fc_out.copy(); tmp["Group"] = g
        all_rows.append(tmp)

        # per-group plot
        fig, ax = plt.subplots()
        ax.plot(y.index, y.values, marker="o", label=f"{g} (obs)")
        ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
        ax.fill_between(fc["Year"], fc["Lo.95"], fc["Hi.95"], alpha=0.14,
                        color=ln.get_color(), label="95% CI (fc)")
        ax.axvline(x=LAST_OBS_YEAR+0.5, linestyle=":", linewidth=2.0)
        ax.set_title(f"{title_prefix}: {g} (≤{LAST_OBS_YEAR}) ARIMA forecast ({FORECAST_START}–{FORECAST_END})")
        ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        style_axes(ax)
        ax.legend(ncols=2)
        plt.tight_layout()
        png_path = out_dir / f"{safe_name(g)}_timeseries_{FORECAST_START}_to_{FORECAST_END}.png"
        plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
        print("   Plot:", png_path)

    if all_rows:
        all_df = pd.concat(all_rows, ignore_index=True)
        all_df.to_csv(out_dir / f"ALL_{safe_name(TITLE_PREFIX.lower())}_{FORECAST_START}_to_{FORECAST_END}.csv", index=False)
        print("Consolidated CSV:", out_dir / f"ALL_{safe_name(TITLE_PREFIX.lower())}_{FORECAST_START}_to_{FORECAST_END}.csv")

def plot_combined(observed_df, out_dir:Path, title: str):
    obs = observed_df.copy()
    obs["Group"] = obs["Group"].replace(
        {r"(?i)non.*metro.*":"Non-metro", r"(?i)^metro$":"Metro"}, regex=True
    )
    obs = obs[obs["Group"].isin(["Metro","Non-metro"])]

    groups = sorted(obs["Group"].unique())
    if not groups:
        print("No groups to plot in combined figure.")
        return

    # read saved forecasts
    fc_map = {}
    for g in groups:
        p = out_dir / f"{safe_name(g)}_forecast_{FORECAST_START}_to_{FORECAST_END}.csv"
        if p.exists():
            fc_map[g] = pd.read_csv(p)

    fig, ax = plt.subplots()
    ci_added = False

    for g in groups:
        sub = (obs[obs["Group"]==g]
               .groupby("Year", as_index=False)["AAMR"].mean()
               .sort_values("Year"))
        sub = sub[sub["Year"] <= LAST_OBS_YEAR]
        ax.plot(sub["Year"], sub["AAMR"], marker="o", label=f"{g} (obs)")
        if g in fc_map:
            fc = fc_map[g]
            ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label=f"{g} (fc)")
            if not ci_added:
                ax.fill_between(fc["Year"], fc["Lo.95"], fc["Hi.95"], alpha=0.12,
                                color=ln.get_color(), label="95% CI (fc)")
                ci_added = True
            else:
                ax.fill_between(fc["Year"], fc["Lo.95"], fc["Hi.95"], alpha=0.12,
                                color=ln.get_color())

    ax.axvline(x=LAST_OBS_YEAR+0.5, linestyle=":", linewidth=2.0)
    ax.set_title(f"{title} (≤{LAST_OBS_YEAR}) ARIMA forecast ({FORECAST_START}–{FORECAST_END})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    ax.margins(x=0.02)
    plt.tight_layout()
    out = out_dir / f"COMBINED_{safe_name(title.lower())}_{FORECAST_START}_to_{FORECAST_END}.png"
    plt.savefig(out, dpi=300, bbox_inches="tight"); plt.close()
    print("Combined plot:", out)

# ---------- driver ----------
def choose_sheets(xls: pd.ExcelFile):
    names = xls.sheet_names
    # Prefer a single tidy sheet if present
    prefer = [s for s in names if s.lower().strip() in ("data final", "data_final", "final", "data")]
    if prefer:
        return prefer
    # Otherwise, take the two split sheets; skip helper sheets
    candidates = []
    for s in names:
        low = s.lower().strip()
        if any(k in low for k in ["join", "graph", "table"]):
            continue
        if any(k in low for k in ["non", "met"]):
            candidates.append(s)
    return candidates or names  # fallback

def main():
    xls = pd.ExcelFile(FILE_PATH)
    sheet_list = choose_sheets(xls)
    print("Sheets selected:", sheet_list)

    all_obs = []
    for nm in sheet_list:
        try:
            obs = load_sheet_allow_groups(FILE_PATH, nm)
            all_obs.append(obs)
        except Exception as e:
            print(f"Skipping sheet '{nm}': {e}")

    if not all_obs:
        print("No usable sheets loaded.")
        return

    observed = pd.concat(all_obs, ignore_index=True)
    print("\nGroups detected for modeling:", sorted(observed['Group'].unique()))

    fit_all_groups_and_save(observed, OUT_DIR, TITLE_PREFIX)
    plot_combined(observed, OUT_DIR, TITLE_PREFIX)

    print("\nDone →", OUT_DIR)

if __name__ == "__main__":
    main()


Sheets selected: ['data final']

Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate' | Group column: 'Variable'
   Group  Year    AAMR
0  Metro  1999  124.32
1  Metro  2000  121.90
2  Metro  2001  119.43
3  Metro  2002  116.85
4  Metro  2003  108.26
5  Metro  2004  101.61
Last observed year forced to: 2020

Groups detected for modeling: ['Metro', 'Non-metro']
[Metro] ARIMA order=(3, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\urbanization\Metro_forecast_2021_to_2043.csv
   Plot: D:\arima project\Task 4\results\urbanization\Metro_timeseries_2021_to_2043.png
[Non-metro] ARIMA order=(2, 2, 3, 'n') → CSV: D:\arima project\Task 4\results\urbanization\Non-metro_forecast_2021_to_2043.csv
   Plot: D:\arima project\Task 4\results\urbanization\Non-metro_timeseries_2021_to_2043.png
Consolidated CSV: D:\arima project\Task 4\results\urbanization\ALL_urbanization_2021_to_2043.csv
Combined plot: D:\arima project\Task 4\results\urbanization\COMBINED_urbanization_2021_to_2043.pn

In [8]:
# -*- coding: utf-8 -*-
# OVERALL: ARIMA forecast to 2043 with bold plots + 95% CI

import re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

# ========= USER PATHS =========
FILE_PATH = Path(r"D:\arima project\Task 4\data\Copy of OVERALL COMLETE DM & STROKE(1).xlsx")
OUT_DIR   = Path(r"D:\arima project\Task 4\results\overall")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# ==============================

# ---------- Plot style ----------
plt.rcParams.update({
    "figure.figsize": (13, 7.6),
    "figure.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "legend.fontsize": 12,
    "legend.frameon": True,
    "legend.title_fontsize": 13,
    "lines.linewidth": 2.6,
    "lines.markersize": 6.5,
})
def style_axes(ax):
    ax.tick_params(axis="both", labelsize=14, width=1.9, length=6)
    for s in ax.spines.values():
        s.set_linewidth(2.0)

# ---------- helpers ----------
def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multirow headers, normalize whitespace, deduplicate names."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in tup if (str(x) != "nan" and str(x).strip() != "")]).strip()
            for tup in df.columns.to_list()
        ]
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    seen, out = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1; out.append(f"{c}.{seen[c]}")
        else:
            seen[c] = 0;  out.append(c)
    df.columns = out
    return df

def _pick_col(df, patterns, required=True, exclude_regex=None):
    low = {c: str(c).strip().lower() for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, lc in low.items():
            if rx.search(lc):
                if exclude_regex and re.search(exclude_regex, lc):
                    continue
                return c
    if required:
        raise KeyError(f"Could not find any of: {patterns}")
    return None

def find_year(df):
    return _pick_col(df, [r"\byear\b", r"^yr$", r"\bcalendar\s*year\b"])

def find_aamr(df):
    """
    Prefer exact 'Age Adjusted Rate' (or similar). If there are multiple,
    pick the rightmost (often the 'overall' rate after sex/race splits).
    """
    cols = []
    for i, c in enumerate(df.columns):
        lc = str(c).lower()
        if ("age" in lc and "adjust" in lc and "rate" in lc):
            cols.append((i, c))
        elif re.search(r"\baamr\b", lc):
            cols.append((i, c))
        elif re.search(r"\bage[-\s]*standard(ized)?\s*rate\b", lc):
            cols.append((i, c))
    if not cols:
        # fallback: any 'rate' with 'adjust'
        for i, c in enumerate(df.columns):
            lc = str(c).lower()
            if ("adjust" in lc and "rate" in lc):
                cols.append((i, c))
    if not cols:
        raise KeyError("Could not find an 'Age Adjusted Rate' column.")
    # choose the last match (often the aggregate)
    cols.sort(key=lambda x: x[0])
    return cols[-1][1]

def choose_sheet(xls: pd.ExcelFile):
    # Prefer sheets that look like final/overall tidy tables
    names = xls.sheet_names
    priority = [
        "overall final", "overall", "data final", "final", "data", "tables", "graph"
    ]
    for p in priority:
        for s in names:
            if s.strip().lower() == p:
                return s
    # otherwise choose the first
    return names[0]

def load_observed_overall(path: Path):
    xls = pd.ExcelFile(path)
    sheet = choose_sheet(xls)
    df = pd.read_excel(xls, sheet_name=sheet)
    df = _flatten_columns(df)

    ycol = find_year(df)
    tcol = find_aamr(df)

    # coerce numeric
    df[ycol] = pd.to_numeric(df[ycol], errors="coerce")
    df[tcol] = pd.to_numeric(df[tcol], errors="coerce")

    tidy = (df[[ycol, tcol]]
            .dropna(subset=[ycol, tcol])
            .groupby(ycol, as_index=False)[tcol].mean()
            .rename(columns={ycol:"Year", tcol:"AAMR"})
            .sort_values("Year"))
    tidy["Year"] = tidy["Year"].astype(int)

    print(f"Sheet: {sheet}")
    print(f"Detected → Year: '{ycol}' | AAMR: '{tcol}'")
    print(tidy.head(6))
    return tidy

def _sanitize_series_for_arima(y: pd.Series):
    y = y.groupby(level=0).mean().sort_index()  # collapse duplicates if any
    full = pd.Index(range(int(y.index.min()), int(y.index.max())+1), name=y.index.name)
    y = y.reindex(full)
    if y.isna().any():
        y = y.interpolate(limit_direction="both")
    return y

def select_arima_order(y):
    best = None
    for d in [0,1,2]:
        for p in range(0,4):
            for q in range(0,4):
                if (p,d,q) == (0,0,0): 
                    continue
                try:
                    trend = "n" if d>0 else "c"
                    res = ARIMA(y, order=(p,d,q), trend=trend,
                                enforce_stationarity=False, enforce_invertibility=False
                               ).fit(method_kwargs={"warn_convergence":False})
                    score = res.aic
                    if (best is None) or (score < best[0]):
                        best = (score, (p,d,q,trend), res)
                except Exception:
                    pass
    if best is None:
        res = ARIMA(y, order=(1,1,0), trend="n",
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(method_kwargs={"warn_convergence":False})
        return (1,1,0,"n"), res
    return best[1], best[2]

def forecast_to(y, end_year, conf=0.95):
    last_year = int(y.index.max())
    steps = max(0, end_year - last_year)
    if steps == 0:
        raise ValueError("Observed series already extends to END_YEAR.")
    order, res = select_arima_order(y)
    fc = res.get_forecast(steps=steps)
    ci = fc.conf_int(alpha=1-conf)
    yrs = np.arange(last_year+1, end_year+1, dtype=int)
    out = pd.DataFrame({
        "Year": yrs,
        "Point.Forecast": fc.predicted_mean.values,
        "Lo.95": ci.iloc[:,0].values,
        "Hi.95": ci.iloc[:,1].values,
        "Order": [f"{order[0]},{order[1]},{order[2]} ({order[3]})"]*steps
    })
    return order, out

def main():
    # 1) Load observed
    obs = load_observed_overall(FILE_PATH)
    last_obs_year = int(obs["Year"].max())
    end_year = 2043
    start_fc = last_obs_year + 1

    # 2) Prepare series
    y = pd.Series(obs["AAMR"].values, index=obs["Year"].astype(int), name="AAMR")
    y = _sanitize_series_for_arima(y)

    # 3) Fit + forecast
    order, fc = forecast_to(y, end_year=end_year, conf=0.95)

    # 4) Save CSV (two decimals)
    csv_path = OUT_DIR / f"Overall_forecast_to_{end_year}.csv"
    fc_out = fc.copy()
    for c in ["Point.Forecast","Lo.95","Hi.95"]:
        fc_out[c] = fc_out[c].round(2)
    fc_out.insert(0, "Series", "Overall")
    fc_out.to_csv(csv_path, index=False)
    print(f"[Overall] order={order}  -> CSV: {csv_path}")

    # 5) Plot
    fig, ax = plt.subplots()
    ax.plot(y.index, y.values, marker="o", label="Overall (obs)")
    ln, = ax.plot(fc["Year"], fc["Point.Forecast"], linestyle="--", marker="o", label="Overall (fc)")
    ax.fill_between(fc["Year"], fc["Lo.95"], fc["Hi.95"], alpha=0.14,
                    color=ln.get_color(), label="95% CI (fc)")
    ax.axvline(x=last_obs_year+0.5, linestyle=":", linewidth=2.0)

    ax.set_title(f"Overall: AAMR observed (≤{last_obs_year}) and ARIMA forecast ({start_fc}–{end_year})")
    ax.set_xlabel("Year"); ax.set_ylabel("AAMR (per 100,000)")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    style_axes(ax)
    ax.legend(ncols=2)
    plt.tight_layout()
    png_path = OUT_DIR / f"Overall_timeseries_to_{end_year}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight"); plt.close()
    print("   Plot:", png_path)

    # 6) Quick console preview
    print("\nObserved last 5:\n", obs.tail())
    print("\nForecast head & tail:\n", fc_out.head(), "\n...\n", fc_out.tail())

if __name__ == "__main__":
    main()


Sheet: data final
Detected → Year: 'Year' | AAMR: 'Age Adjusted Rate Standard Error'
   Year  AAMR
0  1999  0.86
1  2000  0.84
2  2001  0.83
3  2002  0.81
4  2003  0.78
5  2004  0.75
[Overall] order=(1, 0, 0, 'c')  -> CSV: D:\arima project\Task 4\results\overall\Overall_forecast_to_2043.csv
   Plot: D:\arima project\Task 4\results\overall\Overall_timeseries_to_2043.png

Observed last 5:
     Year  AAMR
20  2019  0.47
21  2020  0.53
22  2021  0.54
23  2022  0.49
24  2023  0.46

Forecast head & tail:
     Series  Year  Point.Forecast  Lo.95  Hi.95      Order
0  Overall  2024            0.45   0.41   0.49  1,0,0 (c)
1  Overall  2025            0.44   0.39   0.50  1,0,0 (c)
2  Overall  2026            0.44   0.37   0.50  1,0,0 (c)
3  Overall  2027            0.43   0.36   0.50  1,0,0 (c)
4  Overall  2028            0.42   0.35   0.50  1,0,0 (c) 
...
      Series  Year  Point.Forecast  Lo.95  Hi.95      Order
15  Overall  2039            0.37   0.26   0.47  1,0,0 (c)
16  Overall  2040      